# Z-Image-Turbo-GGUF Dual — Production-style Demo on Kaggle T4 ×2

**Showcase purpose:** reproducible Vue Studio + dual-GPU inference, backed by authenticated private PNG receipts, plus two separate Cloudflare Quick Tunnels.

**Required Kaggle settings:** Accelerator **GPU T4 x2**, Internet **ON** for initial setup, attach the following Kaggle inputs:
- Model: `dangkhoa2016/unsloth-z-image-turbo-gguf` (GGUF Q6_K, v1).
- Dataset: `dangkhoa2016/z-image-turbo-diffusers-support-components`.

**Execution:** Run cells 1–5 sequentially. Gallery and Studio tunnel cells are independent. There are **no automatic demo render jobs** on notebook startup: click Generate in Studio to trigger T4 GPU inference. Two simultaneous different prompts can occupy GPU0 and GPU1, as demonstrated in the prior acceptance. Every Kaggle session has a runtime limit.

**Gallery caution:** the 48-image and 122-image original galleries are **not in this Git repository** and were not present on the verified Kaggle machine. Attach or restore their exact static HTML + assets first. The gallery cell refuses to mislabel other images as 48/122.


## 1 · Clone the source repository

In [ ]:
from pathlib import Path
import subprocess,os
REPO_URL="https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual.git"
PROJECT=Path("/kaggle/working/zimage-dual")
if not (PROJECT/".git").is_dir():
    assert not PROJECT.exists(),f"Refusing to overwrite existing project: {PROJECT}"
    subprocess.run(["git","clone","--depth","1",REPO_URL,str(PROJECT)],check=True)
else:
    print("Existing project checkout preserved; no automatic pull")
assert (PROJECT/"product-studio-vue-cdn-v3/index.html").exists()
print("SOURCE_READY",PROJECT)


## 2 · Verify attached model, components, T4×2 and runtime requirements

In [ ]:
import subprocess,sys,importlib.metadata as md
model=Path("/kaggle/input/models/dangkhoa2016/unsloth-z-image-turbo-gguf/gguf/q6-k/1/z-image-turbo-Q6_K.gguf")
support=Path("/kaggle/input/datasets/dangkhoa2016/z-image-turbo-diffusers-support-components/z-image-turbo-diffusers-support")
assert model.is_file(),f"Attach Kaggle model first: {model}"
assert (support/"text_encoder").is_dir() and (support/"vae").is_dir(),"Attach support dataset first"
gpus=subprocess.check_output(["nvidia-smi","-L"],text=True)
assert gpus.count("Tesla T4")>=2,"Switch Kaggle accelerator to T4 ×2"
print("GPU_T4X2_READY",gpus)
print("GGUF_BYTES",model.stat().st_size)
for key in ("torch","transformers","diffusers","huggingface-hub","torchao","gguf","safetensors","fastapi","uvicorn"):
    try: print(key,md.version(key))
    except md.PackageNotFoundError:print(key,"MISSING")
print("Source vendors (repo):",list(p.name for p in (PROJECT/"production-system").glob("vendor-*")))
print("NOTE: Tensor library versions must match the validated runtime. Install missing packages in the next cell; do not randomly upgrade torch.")


## 3 · Dependencies and GPU runtime preflight (fail closed)

In [ ]:
import importlib.util
needed={"fastapi":"fastapi","uvicorn":"uvicorn","safetensors":"safetensors","transformers":"transformers","diffusers":"diffusers","gguf":"gguf","torchao":"torchao","accelerate":"accelerate","huggingface_hub":"huggingface_hub"}
missing=[pkg for pkg,mod in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing missing Python dependencies:",missing)
    subprocess.run([sys.executable,"-m","pip","install","-q",*missing],check=True)
import torch
assert torch.cuda.device_count()==2,"Expected 2 visible T4 GPUs"
# Prefer vendored code from the validated project snapshot.
VENDORS=tuple(str(PROJECT/"production-system"/v) for v in ("vendor-shim","vendor-hub128","vendor-diffusers041","vendor-torchao015","vendor-gguf"))
os.environ["PYTHONPATH"]=":".join(VENDORS+tuple(filter(None,[os.environ.get("PYTHONPATH","")])))
os.environ["HF_HUB_OFFLINE"]="1"
os.environ["TRANSFORMERS_OFFLINE"]="1"
print("PREFLIGHT_READY", [torch.cuda.get_device_name(i) for i in range(2)])
print("Restart the notebook kernel if the package installer changed already imported libraries.")


## 4 · Start API + Vue Studio (localhost only)

In [ ]:
import secrets,json,time,urllib.request
from pathlib import Path
PROD=PROJECT/"production-system"
RUN=PROD/"b3-studio-demo-live-20261016"
cfgpath=RUN/"local-secrets.json"
if not cfgpath.exists():
    subprocess.run([sys.executable,str(PROD/"b3-demo-studio-20261016/start_local_demo.py")],check=True,env=os.environ.copy())
else:
    subprocess.run([sys.executable,str(RUN/"resume_local_demo.py")],check=True,env=os.environ.copy())
cfg=json.loads(cfgpath.read_text())
assert all(cfg.get(key) for key in ("api_token","studio_password","service_key"))
assert cfgpath.stat().st_mode & 0o077==0,"Unsafe secret permissions"
for port,path in ((18788,"/health"),(18789,"/auth/session")):
    with urllib.request.urlopen(f"http://127.0.0.1:{port}{path}",timeout=10) as res:
        assert res.status==200
print("STUDIO_GATEWAY_READY: http://127.0.0.1:18789/studio/")
print("Service keys are NOT printed")


## 5 · Start continuous T4×2 GPU consumer (no sample image is created)

In [ ]:
import sqlite3
with sqlite3.connect("file:"+str(RUN/"api.sqlite3")+"?mode=ro",uri=True) as db:
    assert db.execute("PRAGMA integrity_check").fetchone()[0]=="ok"
    print("QUEUE_STATE",list(db.execute("select state,count(*) from jobs group by state")))
consumer=subprocess.run([sys.executable,str(RUN/"start_persistent_gpu_consumer.py")],capture_output=True,text=True,env=os.environ.copy())
# Existing worker means idempotent for the notebook rerun; do not spawn duplicate.
if consumer.returncode:
    if "Existing persistent consumer PID" not in (consumer.stdout+consumer.stderr):
        raise RuntimeError((consumer.stdout+consumer.stderr)[-1300:])
    print("GPU consumer already running; keeping existing process")
else:
    print(consumer.stdout.strip())
print("T4×2 DISPATCH READY: new Studio submissions are handled by the existing lifecycle.")
print("GPU 0 encodes prompts in a bounded phase; independent inference workers can occupy both GPU 0 and GPU 1.")


## 6A · Static benchmark galleries — separate Cloudflare HTTPS tunnel

Place both original gallery directories (each containing `index.html` and images) in `/kaggle/working/gallery-48` and `/kaggle/working/gallery-122`. **Do not substitute Pilot32 for either gallery.** This cell runs its own server and tunnel.


In [ ]:
import re,threading,http.server,socketserver,shutil,urllib.request,subprocess,time
from pathlib import Path
GROOT=Path("/kaggle/working/zimage-gallery-showcase")
sources={"gallery-48":Path("/kaggle/working/gallery-48"),"gallery-122":Path("/kaggle/working/gallery-122")}
for name,folder in sources.items():
    assert (folder/"index.html").is_file(),f"Missing {name}/index.html: attach/restore original gallery assets to {folder}"
    images=list(folder.rglob("*.png"))+list(folder.rglob("*.jpg"))+list(folder.rglob("*.webp"))
    expected=int(name.split("-")[1])
    assert len(images)==expected,f"{name} must have {expected} images; found {len(images)}. Verify original gallery!"
GROOT.mkdir(exist_ok=True)
for name,folder in sources.items():
    link=GROOT/name
    if not link.exists(): link.symlink_to(folder,target_is_directory=True)
(GROOT/"index.html").write_text('<!doctype html><html><head><meta charset="utf-8"><title>Z-Image GPU Evidence Galleries</title></head><body style="font:18px system-ui;background:#101827;color:white;padding:3rem"><h1>Z-Image GPU Evidence — Benchmark Galleries</h1><p><a style="color:#93c5fd" href="/gallery-48/">48-image gallery</a></p><p><a style="color:#93c5fd" href="/gallery-122/">122-image gallery</a></p></body></html>')
class StaticHandler(http.server.SimpleHTTPRequestHandler):
    def __init__(self,*args,**kwargs):super().__init__(*args,directory=str(GROOT),**kwargs)
class ReusableServer(socketserver.ThreadingTCPServer):allow_reuse_address=True
try:
    gallserver=ReusableServer(("127.0.0.1",18881),StaticHandler)
    threading.Thread(target=gallserver.serve_forever,daemon=True).start()
except OSError:
    print("Gallery server already listening")
print("GALLERY_SERVER_READY",urllib.request.urlopen("http://127.0.0.1:18881/",timeout=5).status)


## 6B · Open benchmark gallery HTTPS with Cloudflared (independent URL)

In [ ]:
import shutil,urllib.request,re,subprocess,time
from pathlib import Path
cf=Path("/kaggle/working/cloudflared")
if not cf.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",cf)
    cf.chmod(0o700)
def quick_tunnel(name,port):
    logfile=Path(f"/kaggle/working/{name}-cloudflared.log")
    proc=subprocess.Popen([str(cf),"tunnel","--no-autoupdate","--url",f"http://127.0.0.1:{port}"],stdout=logfile.open("ab"),stderr=subprocess.STDOUT,start_new_session=True)
    print("TUNNEL_PID",proc.pid)
    for attempt in range(50):
        text=logfile.read_text(errors="replace")
        urls=re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com",text)
        if urls:return urls[-1]
        if proc.poll() is not None:raise RuntimeError(f"{name} tunnel exited: {text[-1200:]}")
        time.sleep(1)
    raise RuntimeError(f"Tunnel URL not found in {logfile}")
gallery_url=quick_tunnel("gallery",18881)
print("GALLERY_HTTPS:",gallery_url)
print("48:",gallery_url+"/gallery-48/")
print("122:",gallery_url+"/gallery-122/")


## 7 · Open approved Vue Studio demo HTTPS (separate Cloudflare tunnel)

Public login credentials are **demo-only and printed by this cell** as requested. The internal Bearer/service tokens are **never printed**. Do not expose port 18788 directly; tunnel only the password-protected gateway on port 18789.


In [ ]:
assert urllib.request.urlopen("http://127.0.0.1:18789/auth/session",timeout=8).status==200
studio_url=quick_tunnel("studio",18789)
print("======================================")
print("Z-IMAGE STUDIO DEMO")
print("URL:      ",studio_url+"/studio/")
print("Username:  studio")
print("Password: ",cfg["studio_password"])
print("======================================")
print("This is a publicly reachable demo login; do not use it for sensitive workloads.")


## 8 · Acceptance and presentation notes

- Show the gallery 48 and 122 examples **only when original assets have been attached**. Explain quality limitations honestly; historical Pilot32 quality/adherence was NOT_PASS.
- Use the Studio URL to input **two distinct prompts** only when you actually want new GPU images generated. The API queues jobs and GPU0/GPU1 can perform inference concurrently.
- Inspect job statuses, `Total Ns` elapsed time, View image and SVG loading, Download PNG.
- The notebook **does not persist** files across a new Kaggle runtime by itself. Save Version/export a checkpoint before shutdown.
- Cloudflare Quick Tunnels are temporary; both URLs change after restart.
